# SupplySight AI

## Enterprise Procurement Decision Intelligence Platform

# Notebook 04A

# Enterprise Data Integration

---

## Objective

The objective of this notebook is to integrate the cleaned enterprise procurement
dataset with all standardized external datasets and enterprise dimensions to
create a unified analytical dataset for procurement intelligence.

Unlike the previous notebooks, which focused on extracting, cleaning, and
standardizing individual datasets, this notebook combines all internal and
external information into a single enterprise-ready dataset suitable for
analytics, dashboard development, feature engineering, and predictive modeling.

The integration process includes:

- Loading cleaned procurement transactions
- Integrating the Enterprise Country Dimension
- Integrating Commodity Prices
- Integrating Fuel Prices
- Integrating Exchange Rates
- Integrating Enterprise Weather Dimension
- Integrating Enterprise Holiday Dimension
- Validating integration quality
- Exporting the final enterprise procurement dataset

The resulting dataset provides a comprehensive view of procurement transactions
enriched with macroeconomic indicators, environmental conditions, and business
calendar intelligence.

---

## Import Libraries

In [1]:
# ============================================================
# Import Libraries
# ============================================================

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

---

## Configure Project Paths

In [2]:
# ============================================================
# Configure Project Paths
# ============================================================

PROJECT_ROOT = Path.cwd().parent

RAW_DATA = PROJECT_ROOT / "Datasets" / "Raw Datasets"

PROCESSED = PROJECT_ROOT / "Datasets" / "Processed Datasets"

OUTPUT = PROJECT_ROOT / "Datasets" / "Integrated Dataset"

OUTPUT.mkdir(exist_ok=True)

print("=" * 70)
print("PROJECT PATHS")
print("=" * 70)

print("Project :", PROJECT_ROOT)
print("Processed :", PROCESSED)
print("Output :", OUTPUT)

PROJECT PATHS
Project : C:\Users\chavi\Desktop\SupplySight AI – Enterprise Procurement Decision Intelligence Platform
Processed : C:\Users\chavi\Desktop\SupplySight AI – Enterprise Procurement Decision Intelligence Platform\Datasets\Processed Datasets
Output : C:\Users\chavi\Desktop\SupplySight AI – Enterprise Procurement Decision Intelligence Platform\Datasets\Integrated Dataset


---

## Load Processed Datasets

The cleaned procurement dataset, enterprise dimensions, and standardized
external datasets generated in previous notebooks are loaded for integration.

These datasets have already undergone validation, cleaning, and standardization
during the ETL process and are now ready for enterprise-level integration.

In [3]:
# ============================================================
# Load Processed Datasets
# ============================================================

# Clean ERP Dataset

procurement = pd.read_csv(
    PROCESSED / "clean_dataco.csv",
    parse_dates=["order_date"]
)

# Enterprise Country Dimension

dim_country = pd.read_csv(
    PROCESSED / "dim_country.csv",
    dtype={"iso2": str},
    keep_default_na=False
)

# Commodity Prices

commodity = pd.read_csv(
    PROCESSED / "commodity_prices_cleaned.csv",
    parse_dates=["date"]
)

# Fuel Prices

fuel = pd.read_csv(
    PROCESSED / "fuel_prices_cleaned.csv",
    parse_dates=["date"]
)

# Exchange Rates

exchange_inr = pd.read_csv(
    PROCESSED / "exchange_usd_inr_cleaned.csv",
    parse_dates=["date"]
)

exchange_eur = pd.read_csv(
    PROCESSED / "exchange_usd_eur_cleaned.csv",
    parse_dates=["date"]
)

# Enterprise Weather Dimension

dim_weather = pd.read_csv(
    PROCESSED / "dim_weather.csv",
    parse_dates=["date"],
    dtype={"iso2": str},
    keep_default_na=False
)

# Enterprise Holiday Dimension

dim_holiday = pd.read_csv(
    PROCESSED / "dim_holiday.csv",
    parse_dates=["date"],
    dtype={"iso2": str},
    keep_default_na=False
)

print("All processed datasets loaded successfully.")

All processed datasets loaded successfully.


---

## Dataset Validation

Before integration, each dataset is validated to ensure that it satisfies the
minimum quality requirements for enterprise integration.

The validation checks include:

- Number of records
- Number of columns
- Duplicate rows
- Missing values

This step confirms that only clean and standardized datasets are used during the
integration process.

In [4]:
# ============================================================
# Dataset Validation
# ============================================================

datasets = {
    "Procurement": procurement,
    "Country Dimension": dim_country,
    "Commodity Prices": commodity,
    "Fuel Prices": fuel,
    "Exchange USD-INR": exchange_inr,
    "Exchange USD-EUR": exchange_eur,
    "Weather Dimension": dim_weather,
    "Holiday Dimension": dim_holiday
}

summary = []

for name, df in datasets.items():

    summary.append({
        "Dataset": name,
        "Rows": df.shape[0],
        "Columns": df.shape[1],
        "Duplicate Rows": df.duplicated().sum(),
        "Missing Values": df.isnull().sum().sum()
    })

validation = pd.DataFrame(summary)

display(validation)

,Dataset,Rows,Columns,Duplicate Rows,Missing Values
0,Procurement,180519,51,0,0
1,Country Dimension,164,14,0,0
2,Commodity Prices,48,72,0,0
3,Fuel Prices,1461,5,0,0
4,Exchange USD-INR,1461,2,0,0
5,Exchange USD-EUR,1461,2,0,0
6,Weather Dimension,239604,9,0,0
7,Holiday Dimension,239604,8,0,0


---

## Prepare Integration Keys

To ensure consistent joins across all enterprise datasets, the procurement
transactions are prepared with standardized temporal attributes.

These derived keys will be used throughout the integration process to match
internal procurement transactions with external datasets such as commodity
prices, fuel prices, exchange rates, weather observations, and holiday
calendars.

The following integration keys are created:

- Order Date
- Order Year
- Order Month
- Year-Month Key

In [5]:
# ============================================================
# Prepare Integration Keys
# ============================================================

procurement["order_date"] = pd.to_datetime(
    procurement["order_date"]
)

# Daily key for all external dataset joins
procurement["order_day"] = (
    procurement["order_date"]
    .dt.normalize()
)

procurement["order_year"] = (
    procurement["order_day"]
    .dt.year
)

procurement["order_month"] = (
    procurement["order_day"]
    .dt.month
)

procurement["year_month"] = (
    procurement["order_day"]
    .dt.to_period("M")
    .astype(str)
)

print("=" * 70)
print("INTEGRATION KEYS CREATED")
print("=" * 70)

display(
    procurement[
        [
            "order_date",
            "order_day",
            "order_year",
            "order_month",
            "year_month"
        ]
    ].head()
)

INTEGRATION KEYS CREATED


,order_date,order_day,order_year,order_month,year_month
0,2018-01-31 22:56:00,2018-01-31,2018,1,2018-01
1,2018-01-13 12:27:00,2018-01-13,2018,1,2018-01
2,2018-01-13 12:06:00,2018-01-13,2018,1,2018-01
3,2018-01-13 11:45:00,2018-01-13,2018,1,2018-01
4,2018-01-13 11:24:00,2018-01-13,2018,1,2018-01


### Part 1 Completed

The procurement dataset and all enterprise dimensions have been successfully
loaded and validated.

Integration keys have been generated to support consistent joins between the
internal ERP dataset and standardized external datasets.

The next section integrates the **Enterprise Country Dimension**, which serves
as the foundation for subsequent Weather and Holiday integrations through
standardized ISO country codes.

---

# Enterprise Country Dimension Integration

The Enterprise Country Dimension is the first dimension integrated into the
procurement dataset.

The ERP dataset contains country names in multiple languages and formats, making
direct integration with external datasets unreliable. During the ETL process,
all ERP country names were standardized and mapped to ISO country codes through
the Enterprise Country Dimension.

Integrating this dimension provides standardized geographical attributes that
will be used throughout the remaining integration process.

The Country Dimension contributes:

- Country Key
- Standardized Country Name
- ISO Alpha-2 Code
- ISO Alpha-3 Code
- Region
- Sub-region
- Continent
- Capital City
- Representative Weather Reference City

## Prepare Country Lookup Table

Only the attributes required during integration are selected from the
Enterprise Country Dimension to create a lightweight lookup table.

This improves readability while avoiding unnecessary duplication of columns
during the merge process.

In [6]:
# ============================================================
# Prepare Country Lookup
# ============================================================

country_lookup = dim_country[
    [
        "country_key",
        "erp_country",
        "country_name",
        "iso2",
        "iso3",
        "region",
        "subregion",
        "continent",
        "capital",
        "weather_reference_city"
    ]
].copy()

print("=" * 70)
print("COUNTRY LOOKUP")
print("=" * 70)

print(f"Rows : {country_lookup.shape[0]:,}")
print(f"Columns : {country_lookup.shape[1]}")

display(country_lookup.head())

COUNTRY LOOKUP
Rows : 164
Columns : 10


,country_key,erp_country,country_name,iso2,iso3,region,subregion,continent,capital,weather_reference_city
0,1,Afganistán,Afghanistan,AF,AFG,Asia,Southern Asia,AS,Kabul,Kabul
1,2,Albania,Albania,AL,ALB,Europe,Southern Europe,EU,Tirana,Tirana
2,3,Alemania,Germany,DE,DEU,Europe,Western Europe,EU,Berlin,Berlin
3,4,Angola,Angola,AO,AGO,Africa,Sub-Saharan Africa,AF,Luanda,Luanda
4,5,Arabia Saudí,Saudi Arabia,SA,SAU,Asia,Western Asia,AS,Riyadh,Riyadh


## Integrate Country Dimension

The Country Dimension is merged with the procurement dataset using the ERP
country name.

This integration standardizes all geographical information and introduces the
ISO Alpha-2 country code, which becomes the primary geographic integration key
for the Weather and Holiday Dimensions.

In [7]:
# ============================================================
# Merge Country Dimension
# ============================================================

procurement = procurement.merge(
    country_lookup,
    left_on="order_country",
    right_on="erp_country",
    how="left"
)

print("=" * 70)
print("COUNTRY DIMENSION MERGED")
print("=" * 70)

print(f"Rows : {procurement.shape[0]:,}")
print(f"Columns : {procurement.shape[1]}")

COUNTRY DIMENSION MERGED
Rows : 180,519
Columns : 65


## Integration Validation

The Country Dimension integration is validated to ensure that every procurement
transaction has been successfully mapped to a standardized country.

The validation checks include:

- Duplicate transaction count
- Missing country mappings
- Missing ISO codes
- Country coverage

In [8]:
# ============================================================
# Country Integration Validation
# ============================================================

print("=" * 70)
print("COUNTRY INTEGRATION VALIDATION")
print("=" * 70)

print("Rows After Merge")
print(procurement.shape[0])

print("\nDuplicate Rows")
print(procurement.duplicated().sum())

print("\nMissing Country Keys")
print(procurement["country_key"].isnull().sum())

print("\nMissing ISO2 Codes")
print(procurement["iso2"].isnull().sum())

print("\nUnique ERP Countries")
print(procurement["order_country"].nunique())

print("\nMapped ISO Countries")
print(procurement["iso2"].nunique())

COUNTRY INTEGRATION VALIDATION
Rows After Merge
180519

Duplicate Rows
0

Missing Country Keys
0

Missing ISO2 Codes
0

Unique ERP Countries
164

Mapped ISO Countries
164


## Country Coverage Summary

The following table summarizes the geographical coverage after integration.

A successful integration should demonstrate complete coverage of ERP countries
while providing standardized ISO codes and regional classifications for every
procurement transaction.

In [9]:
# ============================================================
# Country Coverage Summary
# ============================================================

country_summary = pd.DataFrame({

    "Metric": [
        "ERP Countries",
        "Standardized Countries",
        "ISO Alpha-2 Codes",
        "Regions",
        "Sub-regions",
        "Continents"
    ],

    "Value": [
        procurement["order_country"].nunique(),
        procurement["country_name"].nunique(),
        procurement["iso2"].nunique(),
        procurement["region"].nunique(),
        procurement["subregion"].nunique(),
        procurement["continent"].nunique()
    ]

})

display(country_summary)

,Metric,Value
0,ERP Countries,164
1,Standardized Countries,164
2,ISO Alpha-2 Codes,164
3,Regions,5
4,Sub-regions,16
5,Continents,7


# Unmatched Country Validation

In [10]:
# ============================================================
# Unmatched Country Validation
# ============================================================

unmatched = procurement[
    procurement["country_key"].isna()
][
    ["order_country"]
].drop_duplicates()

print("=" * 70)
print("UNMATCHED ERP COUNTRIES")
print("=" * 70)

print(f"Count : {len(unmatched)}")

if unmatched.empty:
    print("\nAll ERP countries successfully mapped.")
else:
    display(unmatched)

UNMATCHED ERP COUNTRIES
Count : 0

All ERP countries successfully mapped.


# Dropping the duplicate ERP country column
After the merge, I recommend dropping the duplicate ERP country column from the lookup to keep the integrated dataset tidy:

In [11]:
procurement.drop(columns=["erp_country"], inplace=True)

## Business Observation

The Enterprise Country Dimension successfully standardizes the geographical
information contained within the procurement dataset.

Instead of relying on inconsistent country names, every procurement transaction
is now associated with internationally recognized ISO country codes together
with regional and continental classifications.

This standardized geographic representation establishes a reliable integration
key for the Enterprise Weather and Holiday Dimensions while also enabling
regional procurement analysis, supplier performance evaluation across markets,
and continent-level business intelligence.

---

# Commodity Price Integration

Commodity prices are important macroeconomic indicators that directly influence
procurement costs across multiple industries.

Unlike transactional ERP data, commodity prices are recorded at a monthly level.
Therefore, procurement transactions are enriched using the Year-Month
integration key created earlier.

This integration enables procurement analysts to evaluate purchasing behaviour
under changing commodity market conditions and later supports procurement cost
forecasting and price sensitivity analysis.

## Prepare Commodity Lookup Table

The commodity dataset is transformed into a lookup table containing the monthly
commodity price indicators required for enterprise integration.

A Year-Month key is generated to establish a consistent relationship between
monthly commodity prices and procurement transactions.

In [12]:
# ============================================================
# Prepare Commodity Lookup
# ============================================================

commodity_lookup = commodity.copy()

commodity_lookup["year_month"] = (
    commodity_lookup["date"]
    .dt.to_period("M")
    .astype(str)
)

print("=" * 70)
print("COMMODITY LOOKUP")
print("=" * 70)

print(f"Rows : {commodity_lookup.shape[0]:,}")
print(f"Columns : {commodity_lookup.shape[1]}")

display(commodity_lookup.head())

COMMODITY LOOKUP
Rows : 48
Columns : 73


,date,crude_oil_average,crude_oil_brent,"crude_oil,_dubai","crude_oil,_wti","coal,_australian","coal,_south_african_**",natural_gas_us,"natural_gas,_europe","liquefied_natural_gas,_japan",natural_gas_index,cocoa,"coffee,_arabica","coffee,_robusta","tea,_avg_3_auctions","tea,_colombo","tea,_kolkata","tea,_mombasa",coconut_oil,groundnuts,fish_meal,groundnut_oil_**,palm_oil,palm_kernel_oil,soybeans,soybean_oil,soybean_meal,rapeseed_oil,sunflower_oil,barley,maize,sorghum,"rice,_thai_5%","rice,_thai_25%","rice,_thai_a.1","rice,_viet_namese_5%","wheat,_us_srw","wheat,_us_hrw","banana,_europe","banana,_us",orange,beef_**,chicken_**,lamb_**,"shrimps,_mexican","sugar,_eu","sugar,_us","sugar,_world","tobacco,_us_import_u.v.","logs,_cameroon","logs,_malaysian","sawnwood,_cameroon","sawnwood,_malaysian",plywood,"cotton,_a_index","rubber,_tsr20_**","rubber,_rss3",phosphate_rock,dap,tsp,urea,potassium_chloride_**,aluminum,iron_ore_cfr_spot,copper,lead,tin,nickel,zinc,gold,platinum,silver,year_month
0,2015-01-01,47.11,48.07,45.98,47.27,61.44,59.29,2.97,9.25,16.19,91.06,2.92,4.19,2.16,2.61,3.24,2.28,2.32,1145.71,1392.86,1712.65,1350.33,719.64,1010.48,433.21,796.58,442.64,777.72,870.71,128.14,174.71,215.06,420.0,400.0,418.55,374.42,231.47,248.39,0.89,0.91,0.76,5.30,2.22,7.93,17.42,0.38,0.56,0.34,4922.86,407.61,251.48,726.83,826.71,461.27,1.48,1.42,1.66,119.50,440.25,381.5,305.0,279.50,1814.72,68.23,5830.54,1843.13,19454.12,14849.19,2113.05,1250.75,1242.45,17.24,2015-01
1,2015-02-01,54.79,57.93,55.83,50.61,69.05,63.48,2.85,8.27,14.20,83.52,2.96,3.94,2.17,2.45,3.11,1.84,2.42,1160.50,1350.00,1677.34,1344.82,723.04,1052.37,423.55,762.61,418.25,762.51,825.00,125.39,173.70,217.71,420.0,400.0,417.00,353.50,219.85,237.00,0.94,1.00,0.70,4.88,2.03,7.71,17.48,0.37,0.54,0.32,4910.97,397.50,250.89,734.30,835.21,460.18,1.54,1.43,1.82,121.25,444.13,382.5,305.0,279.50,1817.82,62.75,5729.27,1795.66,18233.91,14573.84,2097.76,1227.08,1197.16,16.79,2015-02
2,2015-03-01,52.83,55.79,54.91,47.78,64.76,59.92,2.80,8.27,13.04,82.16,2.88,3.54,2.03,2.41,3.08,1.42,2.72,1090.68,1350.00,1570.53,1344.82,698.86,1020.80,409.79,749.93,412.36,751.16,804.55,126.31,174.23,227.74,410.0,392.0,411.00,360.74,218.80,230.75,0.93,1.04,0.63,4.72,2.15,7.58,17.45,0.35,0.53,0.29,4961.57,379.19,247.32,717.91,816.57,453.64,1.53,1.43,1.74,123.00,428.80,383.0,295.5,283.40,1773.86,58.05,5939.67,1792.47,17421.91,13755.50,2028.73,1178.63,1138.64,16.24,2015-03
3,2015-04-01,57.54,59.39,58.80,54.44,56.24,58.95,2.58,6.77,10.94,70.94,2.87,3.62,2.03,2.63,3.01,2.13,2.77,1077.50,1339.50,1569.57,1344.82,682.75,979.75,394.73,747.74,398.09,747.72,833.50,129.06,172.05,224.21,399.0,384.0,400.26,354.02,209.70,223.40,0.91,1.03,0.61,4.82,2.31,7.77,16.93,0.35,0.54,0.29,4986.71,377.76,249.06,715.85,814.23,456.83,1.58,1.41,1.70,114.13,409.00,380.0,260.0,299.00,1819.19,52.28,6042.09,2005.36,15900.88,12830.92,2212.72,1198.93,1151.29,16.34,2015-04
4,2015-05-01,62.51,64.56,63.69,59.27,61.19,61.63,2.84,6.68,9.34,72.57,3.10,3.49,1.93,2.89,2.97,2.60,3.09,1131.94,1270.00,1619.84,1344.82,697.35,962.06,388.20,785.13,386.12,769.95,916.05,126.77,166.29,217.38,381.0,368.0,386.53,351.00,200.77,215.32,0.94,0.94,0.61,4.50,2.35,7.48,15.86,0.36,0.54,0.29,5018.01,390.78,246.50,740.67,842.46,452.14,1.61,1.55,1.84,110.50,413.56,380.0,265.0,300.88,1804.04,60.30,6294.78,1991.78,15803.59,13511.34,2281.80,1198.63,1140.40,16.83,2015-05


## Integrate Commodity Prices

Commodity prices are merged with the procurement dataset using the Year-Month
integration key.

Since commodity prices remain constant for all transactions within a particular
month, this approach ensures that every procurement transaction receives the
appropriate macroeconomic context.

In [13]:
# ============================================================
# Merge Commodity Prices
# ============================================================

procurement = procurement.merge(
    commodity_lookup,
    on="year_month",
    how="left"
)

print("=" * 70)
print("COMMODITY DATA MERGED")
print("=" * 70)

print(f"Rows : {procurement.shape[0]:,}")
print(f"Columns : {procurement.shape[1]}")

COMMODITY DATA MERGED
Rows : 180,519
Columns : 136


## Commodity Integration Validation

The merged dataset is validated to ensure that monthly commodity prices have
been successfully integrated into every procurement transaction.

The validation checks include:

- Duplicate transaction count
- Missing commodity values
- Commodity coverage
- Monthly coverage

In [14]:
# ============================================================
# Commodity Integration Validation
# ============================================================

print("=" * 70)
print("COMMODITY INTEGRATION VALIDATION")
print("=" * 70)

print("Rows After Merge")
print(procurement.shape[0])

print("\nDuplicate Rows")
print(procurement.duplicated().sum())

print("\nMissing Commodity Records")

commodity_columns = commodity_lookup.columns.drop(
    ["date", "year_month"]
)

missing_summary = (
    procurement[commodity_columns]
    .isnull()
    .sum()
    .sort_values(ascending=False)
)

print("=" * 70)
print("MISSING VALUES BY COMMODITY")
print("=" * 70)

display(missing_summary.to_frame("Missing Values"))

print("\nAvailable Months")

print(procurement["year_month"].nunique())

COMMODITY INTEGRATION VALIDATION
Rows After Merge
180519

Duplicate Rows
0

Missing Commodity Records
MISSING VALUES BY COMMODITY


,Missing Values
crude_oil_average,0
"sugar,_us",0
"sawnwood,_malaysian",0
"sawnwood,_cameroon",0
"logs,_malaysian",0
"logs,_cameroon",0
"tobacco,_us_import_u.v.",0
"sugar,_world",0
"sugar,_eu",0
"cotton,_a_index",0



Available Months
37


## Commodity Coverage Summary

The following summary provides an overview of the commodity information
available after integration.

This validation confirms that procurement transactions have been successfully
enriched with monthly commodity market indicators.

In [15]:
# ============================================================
# Commodity Coverage Summary
# ============================================================

commodity_columns = commodity_lookup.columns.drop(
    ["date", "year_month"]
)

commodity_summary = pd.DataFrame({

    "Metric": [
        "Commodity Indicators",
        "Monthly Records",
        "Procurement Months"
    ],

    "Value": [
        len(commodity_columns),
        commodity_lookup.shape[0],
        procurement["year_month"].nunique()
    ]

})

display(commodity_summary)

,Metric,Value
0,Commodity Indicators,71
1,Monthly Records,48
2,Procurement Months,37


In [16]:
procurement.drop(
    columns=["date"],
    errors="ignore",
    inplace=True
)

## Business Observation

Monthly commodity prices have been successfully integrated into the procurement
dataset.

Each procurement transaction is now enriched with historical commodity market
conditions corresponding to its transaction month.

These macroeconomic indicators provide valuable context for analysing cost
fluctuations, supplier pricing behaviour, procurement trends, and future price
forecasting models.

---

# Fuel Price Integration

Fuel prices are an important external economic indicator that can influence
transportation costs, logistics expenses, and overall procurement expenditure.

Unlike monthly commodity prices, fuel prices are available at a daily level.

To ensure accurate integration, procurement transactions are matched using the
standardized **Order Day** integration key created during the data preparation
phase.

This integration enriches each procurement transaction with the corresponding
daily fuel price.

## Prepare Fuel Price Lookup

The cleaned fuel price dataset is transformed into a lightweight lookup table.

Only the attributes required for enterprise analytics are retained to avoid
introducing unnecessary metadata into the integrated procurement dataset.

The lookup table contains:

- Date
- Daily Fuel Price

In [17]:
# ============================================================
# Prepare Fuel Price Lookup
# ============================================================

fuel_lookup = fuel[
    [
        "date",
        "fuel_price"
    ]
].copy()

fuel_lookup.rename(
    columns={
        "fuel_price": "daily_fuel_price"
    },
    inplace=True
)

print("=" * 70)
print("FUEL PRICE LOOKUP")
print("=" * 70)

print(f"Rows : {fuel_lookup.shape[0]:,}")
print(f"Columns : {fuel_lookup.shape[1]}")

display(fuel_lookup.head())

FUEL PRICE LOOKUP
Rows : 1,461
Columns : 2


,date,daily_fuel_price
0,2015-01-01,3.137
1,2015-01-02,3.137
2,2015-01-03,3.137
3,2015-01-04,3.137
4,2015-01-05,3.137


## Integrate Fuel Prices

Daily fuel prices are integrated with procurement transactions using the
standardized **Order Day** key.

Using the normalized daily date instead of the transaction timestamp ensures
that every procurement transaction is matched with the corresponding fuel price
record for that calendar day.

In [18]:
# ============================================================
# Merge Fuel Prices
# ============================================================

procurement = procurement.merge(
    fuel_lookup,
    left_on="order_day",
    right_on="date",
    how="left"
)

# Remove lookup date column if present
if "date" in procurement.columns:
    procurement.drop(columns=["date"], inplace=True)

print("=" * 70)
print("FUEL PRICE DATA MERGED")
print("=" * 70)

print(f"Rows : {procurement.shape[0]:,}")
print(f"Columns : {procurement.shape[1]}")

FUEL PRICE DATA MERGED
Rows : 180,519
Columns : 136


## Fuel Price Integration Validation

The merged dataset is validated to ensure that daily fuel prices have been
successfully integrated into procurement transactions.

The validation checks include:

- Duplicate transaction count
- Missing daily fuel prices
- Fuel dataset coverage
- Available fuel dates

In [19]:
# ============================================================
# Fuel Price Integration Validation
# ============================================================

print("=" * 70)
print("FUEL PRICE INTEGRATION VALIDATION")
print("=" * 70)

print("Rows After Merge")
print(procurement.shape[0])

print("\nDuplicate Rows")
print(procurement.duplicated().sum())

print("\nMissing Daily Fuel Prices")
print(procurement["daily_fuel_price"].isnull().sum())

print("\nFuel Records")
print(fuel_lookup.shape[0])

print("\nAvailable Fuel Dates")
print(fuel_lookup["date"].nunique())

FUEL PRICE INTEGRATION VALIDATION
Rows After Merge
180519

Duplicate Rows
0

Missing Daily Fuel Prices
0

Fuel Records
1461

Available Fuel Dates
1461


## Fuel Price Coverage Summary

The following summary provides an overview of the fuel price information
available after integration.

This validation confirms the completeness of the external fuel price data used
to enrich procurement transactions.

In [20]:
# ============================================================
# Fuel Price Coverage Summary
# ============================================================

fuel_summary = pd.DataFrame({

    "Metric": [
        "Fuel Price Records",
        "Unique Fuel Dates",
        "Missing Daily Fuel Prices"
    ],

    "Value": [
        fuel_lookup.shape[0],
        fuel_lookup["date"].nunique(),
        procurement["daily_fuel_price"].isnull().sum()
    ]

})

display(fuel_summary)

,Metric,Value
0,Fuel Price Records,1461
1,Unique Fuel Dates,1461
2,Missing Daily Fuel Prices,0


In [21]:
procurement.drop(
    columns=["date"],
    errors="ignore",
    inplace=True
)

## Business Observation

The procurement dataset has been successfully enriched with historical daily
fuel prices.

By integrating daily fuel market information, procurement transactions now
include an external economic indicator that can support transportation cost
analysis, logistics performance evaluation, procurement cost forecasting, and
market trend analysis.

Using the standardized **Order Day** integration key ensures consistent daily
matching between procurement transactions and external fuel price records.

---

# Exchange Rate Integration (USD-INR & USD-EUR)

Exchange rates are important macroeconomic indicators that influence procurement
costs for organizations purchasing products across international markets.

The procurement dataset is enriched with daily USD-INR and USD-EUR exchange
rates using the standardized **Order Day** integration key.

These indicators enable future analyses of currency fluctuations, procurement
cost trends, supplier pricing behaviour, and international purchasing risks.

## Prepare Exchange Rate Lookup Tables

The cleaned exchange rate datasets are prepared as lookup tables.

Only the daily exchange rate values required for enterprise analytics are
retained to avoid unnecessary attributes during integration.

In [22]:
# ============================================================
# Prepare Exchange Rate Lookup Tables
# ============================================================

exchange_inr_lookup = exchange_inr[
    [
        "date",
        "usd_inr_rate"
    ]
].copy()

exchange_eur_lookup = exchange_eur[
    [
        "date",
        "usd_eur_rate"
    ]
].copy()

print("=" * 70)
print("USD-INR LOOKUP")
print("=" * 70)

display(exchange_inr_lookup.head())

print("=" * 70)
print("USD-EUR LOOKUP")
print("=" * 70)

display(exchange_eur_lookup.head())

USD-INR LOOKUP


,date,usd_inr_rate
0,2015-01-01,63.27
1,2015-01-02,63.27
2,2015-01-03,63.27
3,2015-01-04,63.27
4,2015-01-05,63.34


USD-EUR LOOKUP


,date,usd_eur_rate
0,2015-01-01,1.2015
1,2015-01-02,1.2015
2,2015-01-03,1.2015
3,2015-01-04,1.2015
4,2015-01-05,1.1918


## Integrate Exchange Rates

Daily exchange rates are merged with procurement transactions using the
standardized **Order Day** integration key.

Each procurement transaction receives the USD-INR and USD-EUR exchange rates
corresponding to its transaction date.

In [23]:
# ============================================================
# Merge USD-INR Exchange Rate
# ============================================================

procurement = procurement.merge(
    exchange_inr_lookup,
    left_on="order_day",
    right_on="date",
    how="left"
)

procurement.drop(
    columns=["date"],
    errors="ignore",
    inplace=True
)

# ============================================================
# Merge USD-EUR Exchange Rate
# ============================================================

procurement = procurement.merge(
    exchange_eur_lookup,
    left_on="order_day",
    right_on="date",
    how="left"
)

procurement.drop(
    columns=["date"],
    errors="ignore",
    inplace=True
)

print("=" * 70)
print("EXCHANGE RATES MERGED")
print("=" * 70)

print(f"Rows : {procurement.shape[0]:,}")
print(f"Columns : {procurement.shape[1]}")

EXCHANGE RATES MERGED
Rows : 180,519
Columns : 138


## Exchange Rate Integration Validation

The merged dataset is validated to ensure that daily exchange rates have been
successfully integrated into procurement transactions.

The validation checks include:

- Duplicate transaction count
- Missing USD-INR exchange rates
- Missing USD-EUR exchange rates
- Exchange rate dataset coverage

In [24]:
# ============================================================
# Exchange Rate Integration Validation
# ============================================================

print("=" * 70)
print("EXCHANGE RATE INTEGRATION VALIDATION")
print("=" * 70)

print("Rows After Merge")
print(procurement.shape[0])

print("\nDuplicate Rows")
print(procurement.duplicated().sum())

print("\nMissing USD-INR Rates")
print(procurement["usd_inr_rate"].isnull().sum())

print("\nMissing USD-EUR Rates")
print(procurement["usd_eur_rate"].isnull().sum())

print("\nUSD-INR Records")
print(exchange_inr_lookup.shape[0])

print("\nUSD-EUR Records")
print(exchange_eur_lookup.shape[0])

EXCHANGE RATE INTEGRATION VALIDATION
Rows After Merge
180519

Duplicate Rows
0

Missing USD-INR Rates
0

Missing USD-EUR Rates
0

USD-INR Records
1461

USD-EUR Records
1461


## Exchange Rate Coverage Summary

The following summary provides an overview of the exchange rate information
available after integration.

This validation confirms that procurement transactions have been successfully
enriched with daily currency exchange rates.

In [25]:
# ============================================================
# Exchange Rate Coverage Summary
# ============================================================

exchange_summary = pd.DataFrame({

    "Metric": [
        "USD-INR Records",
        "USD-EUR Records",
        "Missing USD-INR Rates",
        "Missing USD-EUR Rates"
    ],

    "Value": [
        exchange_inr_lookup.shape[0],
        exchange_eur_lookup.shape[0],
        procurement["usd_inr_rate"].isnull().sum(),
        procurement["usd_eur_rate"].isnull().sum()
    ]

})

display(exchange_summary)

,Metric,Value
0,USD-INR Records,1461
1,USD-EUR Records,1461
2,Missing USD-INR Rates,0
3,Missing USD-EUR Rates,0


## Business Observation

The procurement dataset has been successfully enriched with daily USD-INR and
USD-EUR exchange rates.

Integrating exchange rate information enables procurement analysts to evaluate
the impact of currency fluctuations on purchasing costs, supplier pricing, and
international procurement performance.

Using the standardized **Order Day** integration key ensures accurate daily
alignment between procurement transactions and historical exchange rate data.

---

# Weather Dimension Integration

Weather conditions can significantly influence procurement operations,
transportation efficiency, supplier performance, and delivery timelines.

The procurement dataset is enriched using the Enterprise Weather Dimension,
which contains standardized daily weather observations for each country.

Weather records are integrated using a composite key consisting of:

- ISO Alpha-2 Country Code
- Order Day

This approach ensures that every procurement transaction is associated with the
weather conditions experienced in the corresponding country on the transaction
date.

## Prepare Weather Lookup Table

The Enterprise Weather Dimension is transformed into a lightweight lookup table.

Only the weather attributes required for procurement analytics are retained.

The lookup contains:

- ISO Alpha-2 Country Code
- Date
- Average Temperature
- Maximum Temperature
- Minimum Temperature
- Precipitation
- Rainfall
- Snowfall
- Maximum Wind Speed

In [26]:
# ============================================================
# Prepare Weather Lookup
# ============================================================

weather_lookup = dim_weather[
    [
        "iso2",
        "date",
        "avg_temperature",
        "precipitation",
        "rainfall",
        "snowfall",
        "max_wind_speed"
    ]
].copy()

print("=" * 70)
print("WEATHER LOOKUP")
print("=" * 70)

print(f"Rows : {weather_lookup.shape[0]:,}")
print(f"Columns : {weather_lookup.shape[1]}")

display(weather_lookup.head())

WEATHER LOOKUP
Rows : 239,604
Columns : 7


,iso2,date,avg_temperature,precipitation,rainfall,snowfall,max_wind_speed
0,AF,2015-01-01,-0.7,0.0,0.0,0.00,7.4
1,AF,2015-01-02,0.1,0.0,0.0,0.00,7.7
2,AF,2015-01-03,1.6,0.0,0.0,0.00,7.1
3,AF,2015-01-04,2.4,0.0,0.0,0.00,8.4
4,AF,2015-01-05,1.9,0.0,0.0,0.28,8.4


## Integrate Weather Dimension

Weather observations are merged with procurement transactions using the
composite key consisting of the ISO Alpha-2 country code and the standardized
Order Day.

This integration enriches every procurement transaction with the corresponding
daily weather conditions.

In [27]:
# ============================================================
# Merge Weather Dimension
# ============================================================

procurement = procurement.merge(
    weather_lookup,
    left_on=[
        "iso2",
        "order_day"
    ],
    right_on=[
        "iso2",
        "date"
    ],
    how="left"
)

procurement.drop(
    columns=["date"],
    errors="ignore",
    inplace=True
)

print("=" * 70)
print("WEATHER DIMENSION MERGED")
print("=" * 70)

print(f"Rows : {procurement.shape[0]:,}")
print(f"Columns : {procurement.shape[1]}")

WEATHER DIMENSION MERGED
Rows : 180,519
Columns : 143


## Weather Integration Validation

The merged dataset is validated to ensure that weather observations have been
successfully integrated into procurement transactions.

The validation checks include:

- Duplicate transaction count
- Missing weather observations
- Weather dataset coverage
- Country coverage

In [28]:
# ============================================================
# Weather Integration Validation
# ============================================================

print("=" * 70)
print("WEATHER INTEGRATION VALIDATION")
print("=" * 70)

print("Rows After Merge")
print(procurement.shape[0])

print("\nDuplicate Rows")
print(procurement.duplicated().sum())

weather_columns = [

    "avg_temperature",
    "precipitation",
    "rainfall",
    "snowfall",
    "max_wind_speed"

]

missing_weather = (
    procurement[weather_columns]
    .isnull()
    .sum()
)

print("\nMissing Weather Values")

display(
    missing_weather.to_frame("Missing Values")
)

print("\nWeather Records")
print(weather_lookup.shape[0])

print("\nCountries Covered")
print(weather_lookup["iso2"].nunique())

WEATHER INTEGRATION VALIDATION
Rows After Merge
180519

Duplicate Rows
0

Missing Weather Values


,Missing Values
avg_temperature,0
precipitation,0
rainfall,0
snowfall,0
max_wind_speed,0



Weather Records
239604

Countries Covered
164


## Weather Coverage Summary

The following summary provides an overview of the weather information available
after integration.

This validation confirms that procurement transactions have been successfully
enriched with standardized daily weather observations.

In [29]:
# ============================================================
# Weather Coverage Summary
# ============================================================

weather_summary = pd.DataFrame({

    "Metric": [
        "Weather Records",
        "Countries",
        "Missing Average Temperature",
        "Missing Precipitation",
        "Missing Rainfall",
        "Missing Snowfall",
        "Missing Wind Speed"
    ],

    "Value": [
        weather_lookup.shape[0],
        weather_lookup["iso2"].nunique(),
        procurement["avg_temperature"].isnull().sum(),
        procurement["precipitation"].isnull().sum(),
        procurement["rainfall"].isnull().sum(),
        procurement["snowfall"].isnull().sum(),
        procurement["max_wind_speed"].isnull().sum()
    ]

})

display(weather_summary)

,Metric,Value
0,Weather Records,239604
1,Countries,164
2,Missing Average Temperature,0
3,Missing Precipitation,0
4,Missing Rainfall,0
5,Missing Snowfall,0
6,Missing Wind Speed,0


## Business Observation

The procurement dataset has been successfully enriched with standardized daily
weather observations.

Integrating weather conditions enables procurement analysts to investigate the
impact of environmental factors on transportation, supplier performance,
delivery reliability, and procurement efficiency.

The composite integration key consisting of the ISO Alpha-2 country code and
Order Day ensures accurate matching between procurement transactions and
historical weather observations.

---

# Holiday Dimension Integration

Public holidays and weekends can significantly influence procurement operations,
supplier availability, production schedules, logistics performance, and delivery
lead times.

The procurement dataset is enriched using the Enterprise Holiday Dimension,
which contains standardized holiday information for every country and date.

Holiday records are integrated using a composite key consisting of:

- ISO Alpha-2 Country Code
- Order Day

This integration enables future analysis of procurement performance during
public holidays and weekends.

## Prepare Holiday Lookup Table

The Enterprise Holiday Dimension is transformed into a lightweight lookup table.

Only the holiday attributes required for procurement analytics are retained.

The lookup contains:

- ISO Alpha-2 Country Code
- Date
- Holiday Name
- Holiday Indicator
- Weekend Indicator
- Holiday Data Availability

In [30]:
# ============================================================
# Prepare Holiday Lookup
# ============================================================

holiday_lookup = dim_holiday[
    [
        "iso2",
        "date",
        "holiday_name",
        "is_holiday",
        "is_weekend",
        "holiday_source_available"
    ]
].copy()

print("=" * 70)
print("HOLIDAY LOOKUP")
print("=" * 70)

print(f"Rows : {holiday_lookup.shape[0]:,}")
print(f"Columns : {holiday_lookup.shape[1]}")

display(holiday_lookup.head())

HOLIDAY LOOKUP
Rows : 239,604
Columns : 6


,iso2,date,holiday_name,is_holiday,is_weekend,holiday_source_available
0,AF,2015-01-01,No Public Holiday,0,0,1
1,AF,2015-01-02,No Public Holiday,0,0,1
2,AF,2015-01-03,میلاد پیامبر,1,1,1
3,AF,2015-01-04,No Public Holiday,0,1,1
4,AF,2015-01-05,No Public Holiday,0,0,1


## Integrate Holiday Dimension

Holiday information is merged with procurement transactions using the composite
key consisting of the ISO Alpha-2 country code and the standardized Order Day.

This integration enriches every procurement transaction with the corresponding
holiday information for the transaction date.

In [31]:
# ============================================================
# Merge Holiday Dimension
# ============================================================

procurement = procurement.merge(
    holiday_lookup,
    left_on=[
        "iso2",
        "order_day"
    ],
    right_on=[
        "iso2",
        "date"
    ],
    how="left"
)

procurement.drop(
    columns=["date"],
    errors="ignore",
    inplace=True
)

print("=" * 70)
print("HOLIDAY DIMENSION MERGED")
print("=" * 70)

print(f"Rows : {procurement.shape[0]:,}")
print(f"Columns : {procurement.shape[1]}")

HOLIDAY DIMENSION MERGED
Rows : 180,519
Columns : 147


## Holiday Integration Validation

The merged dataset is validated to ensure that holiday information has been
successfully integrated into procurement transactions.

The validation checks include:

- Duplicate transaction count
- Missing holiday information
- Holiday dataset coverage
- Country coverage

In [32]:
# ============================================================
# Holiday Integration Validation
# ============================================================

print("=" * 70)
print("HOLIDAY INTEGRATION VALIDATION")
print("=" * 70)

print("Rows After Merge")
print(procurement.shape[0])

print("\nDuplicate Rows")
print(procurement.duplicated().sum())

holiday_columns = [

    "holiday_name",
    "is_holiday",
    "is_weekend",
    "holiday_source_available"

]

missing_holiday = (
    procurement[holiday_columns]
    .isnull()
    .sum()
)

print("\nMissing Holiday Values")

display(
    missing_holiday.to_frame("Missing Values")
)

print("\nHoliday Records")
print(holiday_lookup.shape[0])

print("\nCountries Covered")
print(holiday_lookup["iso2"].nunique())

HOLIDAY INTEGRATION VALIDATION
Rows After Merge
180519

Duplicate Rows
0

Missing Holiday Values


,Missing Values
holiday_name,0
is_holiday,0
is_weekend,0
holiday_source_available,0



Holiday Records
239604

Countries Covered
164


## Holiday Coverage Summary

The following summary provides an overview of the holiday information available
after integration.

This validation confirms that procurement transactions have been successfully
enriched with standardized holiday and weekend information.

In [33]:
# ============================================================
# Holiday Coverage Summary
# ============================================================

holiday_summary = pd.DataFrame({

    "Metric": [

        "Holiday Records",
        "Countries",
        "Missing Holiday Name",
        "Missing Holiday Flag",
        "Missing Weekend Flag",
        "Missing Holiday Availability Flag"

    ],

    "Value": [

        holiday_lookup.shape[0],
        holiday_lookup["iso2"].nunique(),
        procurement["holiday_name"].isnull().sum(),
        procurement["is_holiday"].isnull().sum(),
        procurement["is_weekend"].isnull().sum(),
        procurement["holiday_source_available"].isnull().sum()

    ]

})

display(holiday_summary)

,Metric,Value
0,Holiday Records,239604
1,Countries,164
2,Missing Holiday Name,0
3,Missing Holiday Flag,0
4,Missing Weekend Flag,0
5,Missing Holiday Availability Flag,0


## Business Observation

The procurement dataset has been successfully enriched with standardized holiday
and weekend information.

Integrating holiday indicators enables procurement analysts to investigate the
impact of public holidays and weekends on supplier performance, delivery delays,
order processing efficiency, and procurement planning.

The composite integration key consisting of the ISO Alpha-2 country code and
Order Day ensures accurate alignment between procurement transactions and
country-specific holiday calendars.

---

# Final Enterprise Validation

The fully integrated procurement dataset is validated to ensure that all
enterprise dimensions and external market indicators have been successfully
merged.

The validation confirms:

- Dataset dimensions
- Duplicate records
- Missing values
- External data integration completeness

This step verifies that the Enterprise Procurement Dataset is ready for feature
engineering and advanced analytics.

In [34]:
# ============================================================
# Final Enterprise Validation
# ============================================================

print("=" * 70)
print("FINAL ENTERPRISE DATASET VALIDATION")
print("=" * 70)

print("Rows")
print(procurement.shape[0])

print("\nColumns")
print(procurement.shape[1])

print("\nDuplicate Rows")
print(procurement.duplicated().sum())

print("\nTotal Missing Values")
print(procurement.isnull().sum().sum())

FINAL ENTERPRISE DATASET VALIDATION
Rows
180519

Columns
147

Duplicate Rows
0

Total Missing Values
0


## Missing Value Validation

The following table summarizes the remaining missing values after integrating
all enterprise dimensions and external datasets.

A successful integration should result in zero missing values across the final
enterprise dataset.

In [35]:
# ============================================================
# Missing Value Validation
# ============================================================

missing_summary = (
    procurement
    .isnull()
    .sum()
    .to_frame("Missing Values")
)

missing_summary = missing_summary[
    missing_summary["Missing Values"] > 0
]

print("=" * 70)
print("REMAINING MISSING VALUES")
print("=" * 70)

display(missing_summary)

REMAINING MISSING VALUES


,Missing Values


## Enterprise Dataset Summary

The following summary provides an overview of the integrated enterprise
procurement dataset.

It confirms that all internal procurement data has been successfully enriched
with external market intelligence and enterprise reference dimensions.

In [36]:
# ============================================================
# Enterprise Dataset Summary
# ============================================================

enterprise_summary = pd.DataFrame({

    "Metric": [
        "Procurement Transactions",
        "Countries",
        "Commodity Indicators",
        "Fuel Records",
        "Weather Records",
        "Holiday Records"

    ],

    "Value": [
        len(procurement),
        procurement["iso2"].nunique(),
        commodity_lookup.shape[1] - 2,
        fuel_lookup.shape[0],
        weather_lookup.shape[0],
        holiday_lookup.shape[0]

    ]

})

display(enterprise_summary)

,Metric,Value
0,Procurement Transactions,180519
1,Countries,164
2,Commodity Indicators,71
3,Fuel Records,1461
4,Weather Records,239604
5,Holiday Records,239604


## Enterprise Data Quality Validation

The following validation confirms that the final enterprise procurement dataset
meets all quality requirements before export.

The validation verifies:

- Dataset dimensions
- Duplicate records
- Missing values
- Country coverage
- Date coverage

In [37]:
# ============================================================
# Enterprise Data Quality Validation
# ============================================================

print("=" * 70)
print("ENTERPRISE DATA QUALITY VALIDATION")
print("=" * 70)

print("Rows")
print(procurement.shape[0])

print("\nColumns")
print(procurement.shape[1])

print("\nDuplicate Rows")
print(procurement.duplicated().sum())

print("\nTotal Missing Values")
print(procurement.isnull().sum().sum())

print("\nCountries")
print(procurement["iso2"].nunique())

print("\nOrder Date Range")
print(
    procurement["order_day"].min(),
    "to",
    procurement["order_day"].max()
)

ENTERPRISE DATA QUALITY VALIDATION
Rows
180519

Columns
147

Duplicate Rows
0

Total Missing Values
0

Countries
164

Order Date Range
2015-01-01 00:00:00 to 2018-01-31 00:00:00


## Rename Enterprise Dataset Columns

Before exporting, the integrated procurement dataset is standardized using
business-friendly column names.

The updated naming convention improves readability, consistency, and usability
for downstream analytics, machine learning, and dashboard development.

In [45]:
for i, col in enumerate(procurement.columns, start=1):
    print(f"{i}. {col}")

1. transaction_type
2. actual_shipping_days
3. scheduled_shipping_days
4. order_profit
5. customer_sales
6. delivery_status
7. late_delivery_risk
8. category_id
9. category_name
10. customer_city
11. customer_country
12. customer_email
13. customer_first_name
14. customer_id
15. customer_last_name
16. customer_password
17. customer_segment
18. customer_state
19. customer_address
20. customer_postal_code
21. department_id
22. department_name
23. latitude
24. longitude
25. market
26. order_city
27. erp_country
28. order_customer_id
29. order_datetime
30. order_id
31. order_product_card_id
32. discount_amount
33. discount_rate
34. order_item_id
35. unit_price
36. profit_ratio
37. quantity
38. gross_sales
39. line_total
40. net_profit
41. order_region
42. order_state
43. order_status
44. product_card_id
45. product_category_id
46. product_image_url
47. product_name
48. list_price
49. product_status
50. shipping_datetime
51. shipping_mode
52. order_day
53. year
54. month
55. year_month
56. 

In [46]:
# ============================================================
# Standardize Enterprise Column Names
# ============================================================

procurement.rename(columns={

    # =============================
    # Shipment
    # =============================
    "type": "transaction_type",
    "days_for_shipping_(real)": "actual_shipping_days",
    "days_for_shipment_(scheduled)": "scheduled_shipping_days",
    "benefit_per_order": "order_profit",
    "sales_per_customer": "customer_sales",

    # =============================
    # Delivery
    # =============================
    "delivery_status": "delivery_status",
    "late_delivery_risk": "late_delivery_risk",

    # =============================
    # Category
    # =============================
    "category_id": "category_id",
    "category_name": "category_name",

    # =============================
    # Customer
    # =============================
    "customer_city": "customer_city",
    "customer_country": "customer_country",
    "customer_email": "customer_email",
    "customer_fname": "customer_first_name",
    "customer_lname": "customer_last_name",
    "customer_password": "customer_password",
    "customer_segment": "customer_segment",
    "customer_state": "customer_state",
    "customer_street": "customer_address",
    "customer_zipcode": "customer_postal_code",

    # =============================
    # Department
    # =============================
    "department_id": "department_id",
    "department_name": "department_name",

    # =============================
    # Geography
    # =============================
    "latitude": "latitude",
    "longitude": "longitude",
    "market": "market",

    # =============================
    # Order
    # =============================
    "order_city": "order_city",
    "order_country": "erp_country",
    "order_customer_id": "customer_order_id",
    "order_date": "order_datetime",
    "order_id": "order_id",

    # =============================
    # Order Item
    # =============================
    "order_item_cardprod_id": "product_card_id_order",
    "order_item_discount": "discount_amount",
    "order_item_discount_rate": "discount_rate",
    "order_item_id": "order_item_id",
    "order_item_product_price": "unit_price",
    "order_item_profit_ratio": "profit_ratio",
    "order_item_quantity": "quantity",
    "order_item_total": "line_total",

    # =============================
    # Sales
    # =============================
    "sales": "gross_sales",
    "order_profit_per_order": "net_profit",

    # =============================
    # Order Geography
    # =============================
    "order_region": "order_region",
    "order_state": "order_state",
    "order_status": "order_status",

    # =============================
    # Product
    # =============================
    "product_card_id": "product_card_id",
    "product_category_id": "product_category_id",
    "product_image": "product_image_url",
    "product_name": "product_name",
    "product_price": "list_price",
    "product_status": "product_status",

    # =============================
    # Shipping
    # =============================
    "shipping_date": "shipping_datetime",
    "shipping_mode": "shipping_mode",

    # =============================
    # Date Keys
    # =============================
    "order_date_key": "order_day",
    "order_year": "year",
    "order_month": "month",
    "year_month": "year_month",

    # =============================
    # Country Dimension
    # =============================
    "country_key": "country_key",
    "country_name": "standard_country",
    "iso2": "country_iso2",
    "iso3": "country_iso3",
    "region": "world_region",
    "subregion": "world_subregion",
    "continent": "continent",
    "capital": "capital_city",
    "weather_reference_city": "weather_city",

    # =============================
    # Fuel
    # =============================
    "daily_fuel_price": "fuel_price",

    # =============================
    # Exchange
    # =============================
    "usd_to_inr": "usd_to_inr",
    "usd_to_eur": "usd_to_eur",

    # =============================
    # Weather
    # =============================
    "avg_temp": "avg_temperature",
    "precipitation": "precipitation",
    "rainfall": "rainfall",
    "snowfall": "snowfall",
    "max_wind_speed": "max_wind_speed",

    # =============================
    # Holiday
    # =============================
    "holiday_name": "holiday_name",
    "is_holiday": "is_holiday",
    "is_weekend": "is_weekend",
    "holiday_data_available": "holiday_data_available"

}, inplace=True)

In [47]:
procurement.columns = (
    procurement.columns
    .str.lower()
    .str.replace(",", "", regex=False)
    .str.replace("%", "pct", regex=False)
    .str.replace("**", "", regex=False)
    .str.replace(".", "", regex=False)
    .str.replace(" ", "_", regex=False)
)

## Final Column Standardization

Before exporting, the integrated procurement dataset is finalized using a
consistent enterprise naming convention.

The final schema follows business-friendly snake_case naming, removes temporary
merge artifacts, and standardizes naming across procurement, geography,
commodity, weather, fuel, exchange rate, and holiday attributes.

This produces a clean analytical dataset suitable for SQL, Power BI, machine
learning, and downstream enterprise reporting.

In [48]:
# ============================================================
# Final Enterprise Column Standardization
# ============================================================

procurement.rename(
    columns={

        # ----------------------------------------------------
        # Order & Customer IDs
        # ----------------------------------------------------
        "customer_order_id": "order_customer_id",
        "product_card_id_order": "order_product_card_id",

        # ----------------------------------------------------
        # Commodity Columns
        # ----------------------------------------------------
        "coal_south_african_": "coal_south_african",
        "groundnut_oil_": "groundnut_oil",
        "beef_": "beef",
        "chicken_": "chicken",
        "lamb_": "lamb",
        "rubber_tsr20_": "rubber_tsr20",
        "potassium_chloride_": "potassium_chloride"

    },
    inplace=True
)

print("=" * 70)
print("FINAL COLUMN STANDARDIZATION COMPLETED")
print("=" * 70)

print(f"Total Columns : {procurement.shape[1]}")

display(procurement.head())

FINAL COLUMN STANDARDIZATION COMPLETED
Total Columns : 147


,transaction_type,actual_shipping_days,scheduled_shipping_days,order_profit,customer_sales,delivery_status,late_delivery_risk,category_id,category_name,customer_city,customer_country,customer_email,customer_first_name,customer_id,customer_last_name,customer_password,customer_segment,customer_state,customer_address,customer_postal_code,department_id,department_name,latitude,longitude,market,order_city,erp_country,order_customer_id,order_datetime,order_id,order_product_card_id,discount_amount,discount_rate,order_item_id,unit_price,profit_ratio,quantity,gross_sales,line_total,net_profit,order_region,order_state,order_status,product_card_id,product_category_id,product_image_url,product_name,list_price,product_status,shipping_datetime,shipping_mode,order_day,year,month,year_month,country_key,standard_country,country_iso2,country_iso3,world_region,world_subregion,continent,capital_city,weather_city,crude_oil_average,crude_oil_brent,crude_oil_dubai,crude_oil_wti,coal_australian,coal_south_african,natural_gas_us,natural_gas_europe,liquefied_natural_gas_japan,natural_gas_index,cocoa,coffee_arabica,coffee_robusta,tea_avg_3_auctions,tea_colombo,tea_kolkata,tea_mombasa,coconut_oil,groundnuts,fish_meal,groundnut_oil,palm_oil,palm_kernel_oil,soybeans,soybean_oil,soybean_meal,rapeseed_oil,sunflower_oil,barley,maize,sorghum,rice_thai_5pct,rice_thai_25pct,rice_thai_a1,rice_viet_namese_5pct,wheat_us_srw,wheat_us_hrw,banana_europe,banana_us,orange,beef,chicken,lamb,shrimps_mexican,sugar_eu,sugar_us,sugar_world,tobacco_us_import_uv,logs_cameroon,logs_malaysian,sawnwood_cameroon,sawnwood_malaysian,plywood,cotton_a_index,rubber_tsr20,rubber_rss3,phosphate_rock,dap,tsp,urea,potassium_chloride,aluminum,iron_ore_cfr_spot,copper,lead,tin,nickel,zinc,gold,platinum,silver,fuel_price,usd_inr_rate,usd_eur_rate,avg_temperature,precipitation,rainfall,snowfall,max_wind_speed,holiday_name,is_holiday,is_weekend,holiday_source_available
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,73,Sporting Goods,Caguas,Puerto Rico,XXXXXXXXX,Cally,20755,Holloway,XXXXXXXXX,Consumer,PR,5365 Noble Nectar Island,725.0,2,Fitness,18.251453,-66.037056,Pacific Asia,Bekasi,Indonesia,20755,2018-01-31 22:56:00,77202,1360,13.110000,0.04,180517,327.75,0.29,1,327.75,314.640015,91.250000,Southeast Asia,Java Occidental,COMPLETE,1360,73,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2018-02-03 22:56:00,Standard Class,2018-01-31,2018,1,2018-01,71,Indonesia,ID,IDN,Asia,South-eastern Asia,AS,Jakarta,Jakarta,66.23,68.99,66.02,63.67,106.45,97.48,3.86,6.66,9.34,84.82,1.95,3.06,1.95,3.1,4.16,2.19,2.96,1393.86,1180.22,1596.54,1433.0,703.45,1265.0,389.53,870.5,375.74,848.38,799.04,121.71,155.84,178.46,442.0,422.0,410.83,380.94,178.34,192.17,0.98,1.13,0.76,4.3,2.07,7.72,12.36,0.4,0.59,0.31,4649.44,426.69,268.3,661.09,751.94,492.12,2.01,1.5,1.72,80.0,360.38,320.0,219.63,215.5,2209.73,76.34,7065.85,2584.09,20696.91,12864.88,3441.52,1331.3,990.12,17.13,3.070,63.58,1.2428,26.1,13.5,13.5,0.0,16.2,No Public Holiday,0,0,1
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,73,Sporting Goods,Caguas,Puerto Rico,XXXXXXXXX,Irene,19492,Luna,XXXXXXXXX,Consumer,PR,2679 Rustic Loop,725.0,2,Fitness,18.279451,-66.037064,Pacific Asia,Bikaner,India,19492,2018-01-13 12:27:00,75939,1360,16.389999,0.05,179254,327.75,-0.80,1,327.75,311.359985,-249.089996,South Asia,Rajastán,PENDING,1360,73,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2018-01-18 12:27:00,Standard Class,2018-01-13,2018,1,2018-01,70,India,IN,IND,Asia,Southern Asia,AS,New Delhi,New Delhi,66.23,68.99,66.02,63.67,106.45,97.48,3.86,6.66,9.34,84.82,1.95,3.06,1.95,3.1,4.16,2.19,2.96,1393.86,1180.22,1596.54,1433.0,703.45,1265.0,389.53,870.5,375.74,848.38,799.04,121.71,155.84,178.46,442.0,422.0,410.83,380.94,178.34,192.17,0.98,1.13,0.76,4.3,2.07,7.72,12.36,0.4,0.59,0.31,4649.44,426.69,268.3,661.09,751.94,492.12,2.01,1.5,1.72,80.0,360.38,320.0,219.63,215.5,2209.73,76.34,7065.85,2584.09,20696.91,12864.88,3441.52,1331.3,990.12

In [49]:
# ============================================================
# Validate Final Schema
# ============================================================

print("=" * 70)
print("FINAL ENTERPRISE SCHEMA")
print("=" * 70)

print("Total Columns :", len(procurement.columns))

print("\nDuplicate Column Names :")
print(procurement.columns.duplicated().sum())

print("\nRemaining Columns Ending with '_' :")

remaining = [
    col
    for col in procurement.columns
    if col.endswith("_")
]

print(remaining)

print("\nFinal Column List")

for i, col in enumerate(procurement.columns, start=1):
    print(f"{i:>3}. {col}")

FINAL ENTERPRISE SCHEMA
Total Columns : 147

Duplicate Column Names :
0

Remaining Columns Ending with '_' :
[]

Final Column List
  1. transaction_type
  2. actual_shipping_days
  3. scheduled_shipping_days
  4. order_profit
  5. customer_sales
  6. delivery_status
  7. late_delivery_risk
  8. category_id
  9. category_name
 10. customer_city
 11. customer_country
 12. customer_email
 13. customer_first_name
 14. customer_id
 15. customer_last_name
 16. customer_password
 17. customer_segment
 18. customer_state
 19. customer_address
 20. customer_postal_code
 21. department_id
 22. department_name
 23. latitude
 24. longitude
 25. market
 26. order_city
 27. erp_country
 28. order_customer_id
 29. order_datetime
 30. order_id
 31. order_product_card_id
 32. discount_amount
 33. discount_rate
 34. order_item_id
 35. unit_price
 36. profit_ratio
 37. quantity
 38. gross_sales
 39. line_total
 40. net_profit
 41. order_region
 42. order_state
 43. order_status
 44. product_card_id
 45. 

In [51]:
# ============================================================
# Final Naming Adjustments
# ============================================================

procurement.rename(
    columns={
        "usd_inr_rate": "usd_to_inr",
        "usd_eur_rate": "usd_to_eur",
        "holiday_source_available": "holiday_data_available"
    },
    inplace=True
)

print("Final naming adjustments completed.")

Final naming adjustments completed.


## Reorder Enterprise Dataset Columns

The final enterprise dataset is reorganized into logical business groups before
export.

The reordered schema improves readability, simplifies downstream analytics, and
aligns with enterprise data warehouse design principles.

Column groups include:

- Transaction Information
- Customer Information
- Product Information
- Order Information
- Geographic Information
- Country Dimension
- Commodity Market Indicators
- External Economic Indicators
- Weather Information
- Holiday Information

In [53]:
# ============================================================
# Reorder Enterprise Dataset Columns
# ============================================================

ordered_columns = [

    # ========================================================
    # Transaction Information
    # ========================================================

    "order_id",
    "order_item_id",
    "transaction_type",
    "order_status",
    "delivery_status",
    "late_delivery_risk",

    "order_datetime",
    "shipping_datetime",
    "order_day",
    "year",
    "month",
    "year_month",

    "actual_shipping_days",
    "scheduled_shipping_days",

    # ========================================================
    # Customer Information
    # ========================================================

    "customer_id",
    "order_customer_id",
    "customer_first_name",
    "customer_last_name",
    "customer_email",
    "customer_password",
    "customer_segment",

    "customer_address",
    "customer_city",
    "customer_state",
    "customer_postal_code",
    "customer_country",

    # ========================================================
    # Product Information
    # ========================================================

    "product_card_id",
    "order_product_card_id",
    "product_category_id",
    "category_id",
    "category_name",
    "department_id",
    "department_name",

    "product_name",
    "product_image_url",
    "product_status",

    # ========================================================
    # Sales & Order Metrics
    # ========================================================

    "quantity",
    "list_price",
    "unit_price",
    "discount_amount",
    "discount_rate",
    "gross_sales",
    "line_total",
    "customer_sales",
    "order_profit",
    "net_profit",
    "profit_ratio",

    # ========================================================
    # Order Geography
    # ========================================================

    "order_city",
    "order_state",
    "order_region",
    "market",
    "erp_country",

    "latitude",
    "longitude",

    # ========================================================
    # Country Dimension
    # ========================================================

    "country_key",
    "standard_country",
    "country_iso2",
    "country_iso3",
    "world_region",
    "world_subregion",
    "continent",
    "capital_city",
    "weather_city",

    # ========================================================
    # Commodity Indicators
    # ========================================================

    "crude_oil_average",
    "crude_oil_brent",
    "crude_oil_dubai",
    "crude_oil_wti",

    "coal_australian",
    "coal_south_african",

    "natural_gas_us",
    "natural_gas_europe",
    "liquefied_natural_gas_japan",
    "natural_gas_index",

    "cocoa",
    "coffee_arabica",
    "coffee_robusta",

    "tea_avg_3_auctions",
    "tea_colombo",
    "tea_kolkata",
    "tea_mombasa",

    "coconut_oil",
    "groundnuts",
    "fish_meal",

    "groundnut_oil",
    "palm_oil",
    "palm_kernel_oil",

    "soybeans",
    "soybean_oil",
    "soybean_meal",

    "rapeseed_oil",
    "sunflower_oil",

    "barley",
    "maize",
    "sorghum",

    "rice_thai_5pct",
    "rice_thai_25pct",
    "rice_thai_a1",
    "rice_viet_namese_5pct",

    "wheat_us_srw",
    "wheat_us_hrw",

    "banana_europe",
    "banana_us",
    "orange",

    "beef",
    "chicken",
    "lamb",
    "shrimps_mexican",

    "sugar_eu",
    "sugar_us",
    "sugar_world",

    "tobacco_us_import_uv",

    "logs_cameroon",
    "logs_malaysian",

    "sawnwood_cameroon",
    "sawnwood_malaysian",

    "plywood",

    "cotton_a_index",

    "rubber_tsr20",
    "rubber_rss3",

    "phosphate_rock",
    "dap",
    "tsp",
    "urea",
    "potassium_chloride",

    "aluminum",
    "iron_ore_cfr_spot",
    "copper",
    "lead",
    "tin",
    "nickel",
    "zinc",
    "gold",
    "platinum",
    "silver",

    # ========================================================
    # External Economic Indicators
    # ========================================================

    "fuel_price",
    "usd_to_inr",
    "usd_to_eur",

    # ========================================================
    # Weather
    # ========================================================

    "avg_temperature",
    "precipitation",
    "rainfall",
    "snowfall",
    "max_wind_speed",

    # ========================================================
    # Holiday
    # ========================================================

    "holiday_name",
    "is_holiday",
    "is_weekend",
    "holiday_data_available"

]

procurement = procurement[ordered_columns]

print("=" * 70)
print("ENTERPRISE DATASET REORDERED")
print("=" * 70)

print("Rows :", procurement.shape[0])
print("Columns :", procurement.shape[1])

display(procurement.head())

ENTERPRISE DATASET REORDERED
Rows : 180519
Columns : 146


,order_id,order_item_id,transaction_type,order_status,delivery_status,late_delivery_risk,order_datetime,shipping_datetime,order_day,year,month,year_month,actual_shipping_days,scheduled_shipping_days,customer_id,order_customer_id,customer_first_name,customer_last_name,customer_email,customer_password,customer_segment,customer_address,customer_city,customer_state,customer_postal_code,customer_country,product_card_id,order_product_card_id,product_category_id,category_id,category_name,department_id,department_name,product_name,product_image_url,product_status,quantity,list_price,unit_price,discount_amount,discount_rate,gross_sales,line_total,customer_sales,order_profit,net_profit,profit_ratio,order_city,order_state,order_region,market,erp_country,latitude,longitude,country_key,standard_country,country_iso2,country_iso3,world_region,world_subregion,continent,capital_city,weather_city,crude_oil_average,crude_oil_brent,crude_oil_dubai,crude_oil_wti,coal_australian,coal_south_african,natural_gas_us,natural_gas_europe,liquefied_natural_gas_japan,natural_gas_index,cocoa,coffee_arabica,coffee_robusta,tea_avg_3_auctions,tea_colombo,tea_kolkata,tea_mombasa,coconut_oil,groundnuts,fish_meal,groundnut_oil,palm_oil,palm_kernel_oil,soybeans,soybean_oil,soybean_meal,rapeseed_oil,sunflower_oil,barley,maize,sorghum,rice_thai_5pct,rice_thai_25pct,rice_thai_a1,rice_viet_namese_5pct,wheat_us_srw,wheat_us_hrw,banana_europe,banana_us,orange,beef,chicken,lamb,shrimps_mexican,sugar_eu,sugar_us,sugar_world,tobacco_us_import_uv,logs_cameroon,logs_malaysian,sawnwood_cameroon,sawnwood_malaysian,plywood,cotton_a_index,rubber_tsr20,rubber_rss3,phosphate_rock,dap,tsp,urea,potassium_chloride,aluminum,iron_ore_cfr_spot,copper,lead,tin,nickel,zinc,gold,platinum,silver,fuel_price,usd_to_inr,usd_to_eur,avg_temperature,precipitation,rainfall,snowfall,max_wind_speed,holiday_name,is_holiday,is_weekend,holiday_data_available
0,77202,180517,DEBIT,COMPLETE,Advance shipping,0,2018-01-31 22:56:00,2018-02-03 22:56:00,2018-01-31,2018,1,2018-01,3,4,20755,20755,Cally,Holloway,XXXXXXXXX,XXXXXXXXX,Consumer,5365 Noble Nectar Island,Caguas,PR,725.0,Puerto Rico,1360,1360,73,73,Sporting Goods,2,Fitness,Smart watch,http://images.acmesports.sports/Smart+watch,0,1,327.75,327.75,13.110000,0.04,327.75,314.640015,314.640015,91.250000,91.250000,0.29,Bekasi,Java Occidental,Southeast Asia,Pacific Asia,Indonesia,18.251453,-66.037056,71,Indonesia,ID,IDN,Asia,South-eastern Asia,AS,Jakarta,Jakarta,66.23,68.99,66.02,63.67,106.45,97.48,3.86,6.66,9.34,84.82,1.95,3.06,1.95,3.1,4.16,2.19,2.96,1393.86,1180.22,1596.54,1433.0,703.45,1265.0,389.53,870.5,375.74,848.38,799.04,121.71,155.84,178.46,442.0,422.0,410.83,380.94,178.34,192.17,0.98,1.13,0.76,4.3,2.07,7.72,12.36,0.4,0.59,0.31,4649.44,426.69,268.3,661.09,751.94,492.12,2.01,1.5,1.72,80.0,360.38,320.0,219.63,215.5,2209.73,76.34,7065.85,2584.09,20696.91,12864.88,3441.52,1331.3,990.12,17.13,3.070,63.58,1.2428,26.1,13.5,13.5,0.0,16.2,No Public Holiday,0,0,1
1,75939,179254,TRANSFER,PENDING,Late delivery,1,2018-01-13 12:27:00,2018-01-18 12:27:00,2018-01-13,2018,1,2018-01,5,4,19492,19492,Irene,Luna,XXXXXXXXX,XXXXXXXXX,Consumer,2679 Rustic Loop,Caguas,PR,725.0,Puerto Rico,1360,1360,73,73,Sporting Goods,2,Fitness,Smart watch,http://images.acmesports.sports/Smart+watch,0,1,327.75,327.75,16.389999,0.05,327.75,311.359985,311.359985,-249.089996,-249.089996,-0.80,Bikaner,Rajastán,South Asia,Pacific Asia,India,18.279451,-66.037064,70,India,IN,IND,Asia,Southern Asia,AS,New Delhi,New Delhi,66.23,68.99,66.02,63.67,106.45,97.48,3.86,6.66,9.34,84.82,1.95,3.06,1.95,3.1,4.16,2.19,2.96,1393.86,1180.22,1596.54,1433.0,703.45,1265.0,389.53,870.5,375.74,848.38,799.04,121.71,155.84,178.46,442.0,422.0,410.83,380.94,178.34,192.17,0.98,1.13,0.76,4.3,2.07,7.72,12.36,0.4,0.59,0.31,4649.44,426.69,268.3,661.09,751.94,492.12,2.01,1.5,1.72,80.0,360.38,320.0,219.63,215.5,2209.73,76.34,7065.85,2584.09,20696.91,12864.88,3441.52,1331.3,990.12,17.13,2.996,63.61,1.2130,13.2,0.0,0.0,0.0,11.2,No

In [54]:
print("=" * 70)
print("FINAL ENTERPRISE DATASET")
print("=" * 70)

print("Rows :", procurement.shape[0])
print("Columns :", procurement.shape[1])

print("\nDuplicate Rows :", procurement.duplicated().sum())
print("Duplicate Columns :", procurement.columns.duplicated().sum())
print("Total Missing Values :", procurement.isnull().sum().sum())

FINAL ENTERPRISE DATASET
Rows : 180519
Columns : 146

Duplicate Rows : 0
Duplicate Columns : 0
Total Missing Values : 0


In [55]:
# ============================================================
# Final Enterprise Dataset Validation
# ============================================================

validation = pd.DataFrame({

    "Validation": [
        "Total Rows",
        "Total Columns",
        "Duplicate Rows",
        "Duplicate Columns",
        "Missing Values",
        "Countries",
        "Order Date Range"
    ],

    "Result": [
        procurement.shape[0],
        procurement.shape[1],
        procurement.duplicated().sum(),
        procurement.columns.duplicated().sum(),
        procurement.isnull().sum().sum(),
        procurement["country_iso2"].nunique(),
        f"{procurement['order_day'].min().date()} to {procurement['order_day'].max().date()}"
    ]

})

print("=" * 70)
print("FINAL ENTERPRISE DATASET VALIDATION")
print("=" * 70)

display(validation)

FINAL ENTERPRISE DATASET VALIDATION


,Validation,Result
0,Total Rows,180519
1,Total Columns,146
2,Duplicate Rows,0
3,Duplicate Columns,0
4,Missing Values,0
5,Countries,164
6,Order Date Range,2015-01-01 to 2018-01-31


## Export Enterprise Dataset

The validated enterprise procurement dataset is exported to the processed
datasets directory.

This dataset will serve as the primary input for the next phase of the project,
which focuses on feature engineering, predictive modeling, and business
intelligence dashboard development.

In [57]:
# ============================================================
# Export Enterprise Dataset
# ============================================================

procurement.to_csv(
    OUTPUT / "enterprise_procurement_dataset.csv",
    index=False
)

print("=" * 70)
print("ENTERPRISE DATASET EXPORTED")
print("=" * 70)

print(OUTPUT / "enterprise_procurement_dataset.csv")

ENTERPRISE DATASET EXPORTED
C:\Users\chavi\Desktop\SupplySight AI – Enterprise Procurement Decision Intelligence Platform\Datasets\Integrated Dataset\enterprise_procurement_dataset.csv


# Notebook Summary

## Objective Achieved

This notebook successfully integrated the enterprise procurement dataset with
all standardized reference dimensions and external market intelligence sources.

## Enterprise Integrations Completed

- Country Dimension
- Commodity Market Indicators
- Daily Fuel Prices
- USD-INR Exchange Rates
- USD-EUR Exchange Rates
- Enterprise Weather Dimension
- Enterprise Holiday Dimension

## Final Dataset Quality

- 180,519 procurement transactions retained
- 146 standardized analytical columns
- 164 standardized countries
- Zero duplicate rows
- Zero duplicate columns
- Zero missing values

## Output Generated

- `enterprise_procurement_dataset.csv`

The exported enterprise dataset serves as the master analytical dataset for the
SupplySight AI – Enterprise Procurement Decision Intelligence Platform and is
ready for feature engineering, predictive analytics, machine learning, and
interactive business intelligence dashboards.